In [0]:
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("merge_keys", "")
dbutils.widgets.text("load_type", "")

landing_timestamp = dbutils.widgets.get("landing_timestamp").strip()
table_name = dbutils.widgets.get("table_name").strip()
merge_keys = dbutils.widgets.get("merge_keys").strip()
load_type = dbutils.widgets.get("load_type").strip().upper() or "FULL"

In [0]:
source_table = f"adb_caresync.bronze.{table_name}"
source_filter = f"landing_timestamp = '{landing_timestamp}'"
source_query_preview = f"SELECT * FROM {source_table} WHERE {source_filter}"

print("=== STEP 1: READ SOURCE DATA ===")
print(f"Source table           : {source_table}")
print(f"Landing timestamp      : {landing_timestamp}")
print("Source query prepared  :")
print(source_query_preview)

bronze_df = spark.read.table(source_table).filter(source_filter)

print("Source DataFrame created successfully")

In [0]:
from pyspark.sql import functions as F

print("=== STEP 2: PREPARE SOURCE DATAFRAME ===")

process_timestamp = F.current_timestamp()
target_table = f"adb_caresync.silver.{table_name}"
merge_key_list = [key.strip() for key in merge_keys.split(",") if key.strip()]

print(f"Target table           : {target_table}")
print(f"Requested load_type    : {load_type}")
print(f"Merge keys raw input   : {merge_keys}")
print(f"Parsed merge keys      : {merge_key_list}")
print(f"Source columns         : {bronze_df.columns}")
print("Adding audit columns   : insert_timestamp, update_timestamp")

updated_df = (
    bronze_df
    .withColumn("insert_timestamp", process_timestamp)
    .withColumn("update_timestamp", process_timestamp)
)

print(f"Prepared output columns: {updated_df.columns}")

In [0]:
print("=== STEP 3: WRITE TO TARGET TABLE ===")
target_exists = spark.catalog.tableExists(target_table)

print(f"Target table exists    : {target_exists}")
print(f"Load type selected     : {load_type}")

if load_type == "FULL":
    print("Action                 : FULL load - overwrite target table")
    print(f"Write command          : updated_df.write.mode('overwrite').saveAsTable('{target_table}')")
    updated_df.write.mode("overwrite").saveAsTable(target_table)
    print("FULL load completed successfully")
elif load_type == "APPEND":
    write_mode = "append" if target_exists else "overwrite"
    print("Action                 : APPEND load")
    print(f"Resolved write mode    : {write_mode}")
    print(f"Write command          : updated_df.write.mode('{write_mode}').saveAsTable('{target_table}')")
    updated_df.write.mode(write_mode).saveAsTable(target_table)
    print("APPEND load completed successfully")
elif load_type == "MERGE":
    print("Action                 : MERGE load")

    if not merge_key_list:
        raise ValueError("merge_keys must be provided when load_type is MERGE")

    print(f"Merge keys received    : {merge_key_list}")
    missing_merge_keys = [key for key in merge_key_list if key not in updated_df.columns]
    if missing_merge_keys:
        raise ValueError(f"merge_keys not found in source data: {', '.join(missing_merge_keys)}")

    print("Merge key validation   : passed")

    if not target_exists:
        print("Target table missing   : creating table with initial overwrite load")
        print(f"Write command          : updated_df.write.mode('overwrite').saveAsTable('{target_table}')")
        updated_df.write.mode("overwrite").saveAsTable(target_table)
        print("Initial target creation completed successfully")
    else:
        source_view = f"{table_name}_merge_source"
        updated_df.createOrReplaceTempView(source_view)
        print(f"Temporary source view  : {source_view}")

        merge_condition = " AND ".join(
            [f"target.`{key}` <=> source.`{key}`" for key in merge_key_list]
        )
        update_assignments = ", ".join(
            [
                f"target.`{column}` = source.`{column}`"
                for column in updated_df.columns
                if column not in {"insert_timestamp", "update_timestamp"}
            ]
            + ["target.`update_timestamp` = current_timestamp()"]
        )
        insert_columns = ", ".join([f"`{column}`" for column in updated_df.columns])
        insert_values = ", ".join([f"source.`{column}`" for column in updated_df.columns])

        print("Merge condition built  :")
        print(merge_condition)
        print("Update assignments     :")
        print(update_assignments)
        print("Insert columns         :")
        print(insert_columns)
        print("Insert values          :")
        print(insert_values)

        merge_sql = f"""
            MERGE INTO {target_table} AS target
            USING {source_view} AS source
            ON {merge_condition}
            WHEN MATCHED THEN UPDATE SET {update_assignments}
            WHEN NOT MATCHED THEN INSERT ({insert_columns})
            VALUES ({insert_values})
        """

        print("Merge SQL prepared     :")
        print(merge_sql)

        spark.sql(merge_sql)
        print("MERGE load completed successfully")
else:
    raise ValueError(f"Unsupported load_type '{load_type}'. Use FULL, APPEND, or MERGE.")

In [0]:
print("=== STEP 4: VALIDATE OUTPUT ===")
print("Counting processed records...")
record_count = updated_df.count()

print(f"Final load_type        : {load_type}")
print(f"Processed row count    : {record_count}")
print(f"Target table           : {target_table}")
print("Notebook execution completed successfully")

In [0]:
dbutils.notebook.exit(str(record_count))